# MarathiSense GPU training on Kaggle

Before **Run All**: use Kaggle **Add Input** to attach (1) the project bundle ZIP and (2) the Mendeley dataset. In notebook settings select **GPU** and enable **Internet**. The project ZIP is not the data.

If Kaggle shows more than one data file, set `DATA_FILE` in the setup cell to the full path of the JSON file containing the Marathi sentence–gloss rows.

In [ ]:
from pathlib import Path
import sys, zipfile
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

# Finds the project by its contents, not by Kaggle's generated dataset/file name.
project_scripts = list(INPUT.rglob('scripts/kaggle_run.py'))
if len(project_scripts) == 1:
    PROJECT = project_scripts[0].parent.parent
else:
    matches = []
    for archive in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as zf:
                if any(n.replace('\\', '/').endswith('MarathiSense/scripts/kaggle_run.py') for n in zf.namelist()):
                    matches.append(archive)
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError('Could not identify exactly one project ZIP/folder. Files under /kaggle/input:\n' + '\n'.join(str(p) for p in INPUT.rglob('*') if p.is_file()))
    with zipfile.ZipFile(matches[0]) as zf:
        zf.extractall(WORK)
    PROJECT = WORK / 'MarathiSense'
assert (PROJECT / 'scripts/kaggle_run.py').is_file(), f'Project runner missing under {PROJECT}'

TABLES = sorted(p for p in INPUT.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.tsv','.json','.jsonl','.xlsx','.parquet'})
print('Project:', PROJECT)
print('Attached data files:')
print('\n'.join(f'  {p}' for p in TABLES) if TABLES else '  No supported data files found')
DATA_FILE = str(TABLES[0]) if len(TABLES) == 1 else None  # If there are multiple, paste the intended full path here.

In [ ]:
# Keep Kaggle's CUDA-enabled torch; install only the project's remaining dependencies.
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-kaggle.txt')], check=True)
# Kaggle may preinstall torchao 0.10, which current PEFT rejects. It is optional for this LoRA run.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. In Notebook settings set Accelerator to GPU, restart the session, and run again.')
print('GPU ready:', torch.cuda.get_device_name(0))
if DATA_FILE is None:
    raise RuntimeError('Several supported data files are attached. Set DATA_FILE above to the full path of the intended Mendeley JSON/table, then rerun this cell.')
print('Selected data file:', DATA_FILE)

In [ ]:
# This validates the dataset schema, prepares the data, trains the baseline and GPU model, and saves reports.
subprocess.run([sys.executable, str(PROJECT / 'scripts/kaggle_run.py'), '--input', DATA_FILE, '--output-root', '/kaggle/working/marathisense'], check=True)

## Save your model

Training outputs are in `/kaggle/working/marathisense`. After the run finishes, choose **Save Version** and enable notebook outputs to preserve the checkpoints, audit, and metrics.